In [1]:
import datetime
import logging  # 追加：ログ制御用
import warnings  # 追加：警告非表示用
import pandas as pd
import yfinance as yf
from IPython.display import display, HTML

# 警告メッセージ（FutureWarningなど）を非表示にする
warnings.filterwarnings("ignore")

# yfinanceの内部エラー出力を非表示にする
yf_logger = logging.getLogger('yfinance')
yf_logger.setLevel(logging.CRITICAL)

# Define the lookback periods (in business days) for performance calculation.
LOOKBACK_DAYS = {
    "1 Day (%)": -2,
    "3 Days (%)": -4,
    "1 Week (%)": -6,
    "1 Month (%)": -21,
    "6 Months (%)": -126,
    "1 Year (%)": -252,
}

# =====================================================================
# DATA PROCESSING FUNCTION (Supports Market Segmentation)
# =====================================================================
def show_market_performances(market_dict):
    today = datetime.date.today()
    start_date = today - datetime.timedelta(days=60)
    
    # Format numerical values to 2 decimal places
    format_config = {"Latest Price": "{:,.2f}"}
    for column_name in LOOKBACK_DAYS.keys():
        format_config[column_name] = "{:+.2f}%"
        
    # ハイライト用の条件関数
    def highlight_extreme_values(val):
        if pd.isna(val):
            return ''
        if val >= 10:
            return 'background-color: #ffcccc; color: #cc0000; font-weight: bold;'
        elif val <= -10:
            return 'background-color: #cce5ff; color: #004085; font-weight: bold;'
        return ''
        
    # Process each market section
    for market_name, ticker_dict in market_dict.items():
        records = []
        
        # Display market header
        display(HTML(f"<h3 style='margin-top: 20px; color: #333;'>■ {market_name} Market</h3>"))
        
        for name, ticker in ticker_dict.items():
            try:
                # 【修正】 ignore_tz=True を追加してタイムゾーン警告を抑制
                df = yf.download(ticker, start=start_date, end=today, progress=False, ignore_tz=True)
                
                if df.empty:
                    # 元のコードにあったWarning表示も邪魔ならコメントアウトしてください
                    # print(f"[Warning] No data found for: {name} ({ticker})")
                    continue
                    
                prices = df["Close"].squeeze()
                latest_price = prices.iloc[-1]
                
                row = {
                    "Name": name,
                    "Ticker": ticker,
                    "Latest Price": latest_price
                }
                
                for label, index in LOOKBACK_DAYS.items():
                    if len(prices) >= abs(index):
                        past_price = prices.iloc[index]
                    else:
                        past_price = prices.iloc[0]
                    
                    row[label] = ((latest_price - past_price) / past_price) * 100
                    
                records.append(row)
                
            except Exception as e:
                # エラー時のprintも非表示にしたい場合はコメントアウトしてください
                pass
        
        if records:
            df_performance = pd.DataFrame(records)
            
            # .map を使用
            styled_df = (df_performance.style
                         .format(format_config)
                         .map(highlight_extreme_values, subset=list(LOOKBACK_DAYS.keys()))
                         .hide(axis="index"))
            
            display(styled_df)
        else:
            print("No data available for this market.")

In [2]:
# =====================================================================
# CONFIGURATION & EXECUTION (Segmented by Market)
# =====================================================================
MARKET_TICKERS = {
    "United States": {
        "S&P 500": "^GSPC",
        "NASDAQ Composite": "^IXIC",
        "Apple": "AAPL",
        "NVIDIA": "NVDA",
        "Microsoft": "MSFT",
    },
    "Japan": {
        "Nikkei 225": "^N225",
        "Toyota Motor": "7203.T",
        "Sony Group": "6758.T",
        "Mitsubishi UFJ Financial": "8306.T",
    },
    "Singapore": {
        "STI Index": "^STI",
        "DBS Group": "D05.SI",
        "UOB": "U11.SI",
        "Singtel": "Z74.SI",
        "CapitaLand Ascendas REIT": "A17U.SI",
    },
    "Japan Stock": {
        "小松製作所 (6301)": "6301.T",
        "ニデック (6594)": "6594.T",
        "未来工業 (7931)": "7931.T",
        "東部ネットワーク (9036)": "9036.T",
        "ニトリHD (9843)": "9843.T",
        "MRK HD (9980)": "9980.T",
    },
    "US Stock (ETF)": {
        "iシェアーズ コア米国総合債券ETF (AGG)": "AGG"
    }
}

# Run and display the segmented tables
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
S&P 500,^GSPC,"7,483.24",+0.00%,+0.58%,+1.71%,-0.93%,+3.92%,+3.92%
NASDAQ Composite,^IXIC,"25,832.67",-0.80%,+0.05%,+1.87%,-3.80%,+3.05%,+3.05%
Apple,AAPL,308.63,+4.84%,+9.54%,+12.17%,-0.53%,+11.59%,+11.59%
NVIDIA,NVDA,194.83,-1.39%,-0.07%,-0.46%,-9.17%,-1.72%,-1.72%
Microsoft,MSFT,390.49,+1.62%,+5.95%,+10.67%,-8.62%,-5.39%,-5.39%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Nikkei 225,^N225,"68,733.15",-2.47%,-1.06%,-5.02%,+1.87%,+9.39%,+9.39%
Toyota Motor,7203.T,"2,793.00",+2.51%,+0.76%,+3.43%,-1.60%,-6.21%,-6.21%
Sony Group,6758.T,"3,330.00",+2.46%,+0.94%,+4.16%,-5.93%,+6.39%,+6.39%
Mitsubishi UFJ Financial,8306.T,"3,306.00",+1.82%,+2.96%,+2.26%,+4.32%,+15.47%,+15.47%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
STI Index,^STI,"5,217.15",+1.08%,+0.16%,-0.03%,+2.95%,+5.95%,+5.95%
DBS Group,D05.SI,66.43,+1.53%,+0.61%,+0.54%,+3.57%,+14.95%,+14.95%
UOB,U11.SI,40.07,+1.06%,+0.43%,+0.40%,+4.59%,+10.78%,+10.78%
Singtel,Z74.SI,4.46,+0.45%,+0.45%,+1.36%,+3.24%,-4.50%,-4.50%
CapitaLand Ascendas REIT,A17U.SI,2.48,+0.81%,-2.36%,-1.98%,-0.80%,-1.20%,-1.20%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
小松製作所 (6301),6301.T,"6,390.00",+2.09%,+1.14%,+2.24%,-8.12%,-2.81%,-2.81%
ニデック (6594),6594.T,"2,678.00",-2.51%,+2.61%,+0.90%,-6.49%,+3.28%,+3.28%
未来工業 (7931),7931.T,"3,125.00",+1.30%,+1.13%,+3.31%,+2.97%,+1.13%,+1.13%
東部ネットワーク (9036),9036.T,"1,258.00",+0.00%,-0.32%,+1.70%,-3.60%,+5.10%,+5.10%
ニトリHD (9843),9843.T,"2,353.50",+1.88%,-5.20%,-1.81%,-14.09%,+4.53%,+4.53%
MRK HD (9980),9980.T,93.00,+0.00%,+2.20%,+2.20%,-6.06%,-5.10%,-5.10%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
iシェアーズ コア米国総合債券ETF (AGG),AGG,98.61,+0.11%,-0.43%,-0.31%,+0.45%,+0.69%,+0.69%


In [3]:
MARKET_TICKERS = {
    "Singapore Healthcare": {
        # --- 病院経営・総合クリニック ---
        "Raffles Medical Group (ラッフルズ医療)": "BSL.SI",  # 【修正】R01.SI から変更
        "IHH Healthcare (マウント・エリザベス等経営)": "Q0F.SI",  # 【修正】I01.SI から変更
        "Thomson Medical Group (トムソン医療)": "A50.SI",
        
        # --- 医療不動産（REIT / 病院・介護施設） ---
        "Parkway Life REIT (パークウェイ・ライフ)": "C2PU.SI",
        "First REIT (ファースト・リート)": "AW9U.SI",
        
        # --- 専門医療・歯科クリニックチェーン ---
        "Q&M Dental Group (歯科大手チェーン)": "QC7.SI",
        "Alliance Healthcare Group (総合診療・クリニック)": "MIJ.SI",  # 【修正】1D8から変更
        "Medinex Limited (医療サポート・クリニック運営)": "OTX.SI",
        
        # --- 医療用消耗品・製薬 ---
        "Medtecs International (医療用消耗品)": "546.SI",
        "Hyphens Pharma (医薬品・皮膚科製品)": "1J5.SI",
        "Riverstone": "AP4.SI"
    }
}
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Raffles Medical Group (ラッフルズ医療),BSL.SI,0.92,+1.10%,+0.55%,-1.08%,-2.65%,-5.13%,-5.13%
IHH Healthcare (マウント・エリザベス等経営),Q0F.SI,2.73,-0.36%,-0.36%,-0.73%,-2.50%,-2.85%,-2.85%
Thomson Medical Group (トムソン医療),A50.SI,0.05,+0.00%,+1.92%,+0.00%,-3.64%,-7.02%,-7.02%
Parkway Life REIT (パークウェイ・ライフ),C2PU.SI,4.08,+0.99%,+0.00%,+0.00%,+3.03%,+1.49%,+1.49%
First REIT (ファースト・リート),AW9U.SI,0.23,+2.22%,+0.00%,+2.22%,+0.00%,-4.17%,-4.17%
Q&M Dental Group (歯科大手チェーン),QC7.SI,0.55,+0.00%,-0.90%,-3.51%,-7.56%,-5.17%,-5.17%
Alliance Healthcare Group (総合診療・クリニック),MIJ.SI,0.16,+0.00%,+2.52%,+2.52%,-0.61%,+0.00%,+0.00%
Medinex Limited (医療サポート・クリニック運営),OTX.SI,0.22,+0.00%,-2.22%,-2.22%,+2.33%,-4.35%,-4.35%
Medtecs International (医療用消耗品),546.SI,0.12,-0.86%,-0.86%,-0.86%,-6.50%,-3.36%,-3.36%
Hyphens Pharma (医薬品・皮膚科製品),1J5.SI,0.34,-1.43%,-1.43%,-1.43%,+2.99%,+1.47%,+1.47%


In [4]:
MARKET_TICKERS = {
"Singapore Food & Agribusiness": {
        # --- 国際的アグリビジネス（巨大穀物・商社） ---
        "Wilmar International (ウィルマー・世界最大級のパーム油)": "F34.SI",
        "Olam Group (オラム・ココア、コーヒー、ナッツ等大手)": "VC2.SI",
        "Olam Agri Holdings (オラムのアグリ部門 ※上場状況による)": "OAG.SI",
        
        # --- パーム油・農園経営（プランテーション） ---
        "Golden Agri-Resources (ゴールデン・アグリ・パーム油大手)": "E5H.SI",
        "First Resources (ファースト・リソーシズ・パーム油生産)": "EB5.SI",
        "Bumitama Agri (ブミタマ・アグリ・インドネシア農園)": "P8D.SI",
        
        # --- 食品製造・流通・飲料 ---
        "Fraser and Neave (F&N・飲料・乳製品大手)": "F99.SI",
        "Japfa (ジャプファ・乳肉製品、畜産大手)": "UD2.SI",
        "Delfi Limited (デルフィ・チョコレート製造・流通)": "P34.SI",
        "Sheng Siong Group (シェンシオン・食品スーパー大手)": "OV8.SI"
    }
}
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Wilmar International (ウィルマー・世界最大級のパーム油),F34.SI,3.70,+1.93%,+0.82%,-0.27%,+6.02%,-0.54%,-0.54%
Olam Group (オラム・ココア、コーヒー、ナッツ等大手),VC2.SI,1.21,+0.83%,+0.83%,-2.42%,-3.20%,+2.54%,+2.54%
Golden Agri-Resources (ゴールデン・アグリ・パーム油大手),E5H.SI,0.27,+1.89%,+0.00%,+1.89%,-5.26%,-14.60%,-14.60%
First Resources (ファースト・リソーシズ・パーム油生産),EB5.SI,3.28,+6.49%,+3.47%,+4.13%,+20.15%,-5.55%,-5.55%
Fraser and Neave (F&N・飲料・乳製品大手),F99.SI,1.43,+0.00%,+0.70%,+0.00%,-0.69%,+1.05%,+1.05%
Delfi Limited (デルフィ・チョコレート製造・流通),P34.SI,0.91,+1.68%,+1.11%,+1.68%,-5.21%,-15.56%,-15.56%
Sheng Siong Group (シェンシオン・食品スーパー大手),OV8.SI,3.33,+4.39%,+2.78%,+2.78%,+9.54%,+9.54%,+9.54%
